# NRMSE cut explorer — lp_fit_align

Interactively tune the NRMSE threshold and inspect its effect. Reads only the fit
checkpoints (a few MB per zip) — **never touches the 120 GB raw cache, never refits**;
editing a parameter and rerunning a cell takes seconds.

Usage: edit `DET` and `NRMSE_MAX` in the 2nd cell, then Run All (or rerun the cells below).

In [1]:
import os, pickle, sys
import numpy as np
import matplotlib.pyplot as plt

SCRIPTS = '/users/9/li004628/urop/snolab/lp_fit_align/scripts'
sys.path.insert(0, SCRIPTS)
from lp_fit_align import ALL_CHANS, CKPT_DIR, RISE_REF_IDX, SAMPLERATE, X_FULL, two_exp_free_pt

def load_fits(det):
    """All fit params of one zip from checkpoints -> {chan: list of dicts}."""
    d = os.path.join(CKPT_DIR, f'zip{det}')
    out = {c: [] for c in ALL_CHANS}
    for f in sorted(os.listdir(d)):
        if not f.endswith('_fit.pkl'):
            continue
        fits = pickle.load(open(os.path.join(d, f), 'rb'))['fits']
        for c in ALL_CHANS:
            out[c] += [fp for fp in (fits.get(c) or []) if fp is not None and fp['fit_ok']]
    return out
print('ready')

ready


In [2]:
# ==== EDIT HERE, then rerun the cells below ====
DET       = 7      # zip number
NRMSE_MAX = 0.4    # cut threshold to try

fits = load_fits(DET)
print(f'zip{DET}: fit_ok events per channel:',
      {c: len(v) for c, v in fits.items() if v})

zip7: fit_ok events per channel: {'PAS1': 2099, 'PBS1': 2008, 'PCS1': 2087, 'PDS1': 2081, 'PES1': 2075, 'PFS1': 1994, 'PAS2': 2095, 'PBS2': 2077, 'PCS2': 2075, 'PDS2': 2070, 'PES2': 2071}


In [3]:
# pass-rate table: events surviving this threshold per channel
print(f'zip{DET}  NRMSE <= {NRMSE_MAX}')
print(f'{"chan":6} {"fit_ok":>8} {"pass":>8} {"pass%":>7}')
for c in ALL_CHANS:
    v = fits[c]
    if not v:
        continue
    n = len(v)
    npass = sum(1 for fp in v if fp['nrmse'] <= NRMSE_MAX)
    print(f'{c:6} {n:>8} {npass:>8} {100*npass/n:>6.1f}%')

zip7  NRMSE <= 0.4
chan     fit_ok     pass   pass%
PAS1       2099     1912   91.1%
PBS1       2008     1931   96.2%
PCS1       2087     1931   92.5%
PDS1       2081     1923   92.4%
PES1       2075     1927   92.9%
PFS1       1994     1927   96.6%
PAS2       2095     1847   88.2%
PBS2       2077     1930   92.9%
PCS2       2075     1930   93.0%
PDS2       2070     1462   70.6%
PES2       2071     1924   92.9%


In [4]:
# NRMSE distribution (log axis) + cut line: where the cut sits in the bimodal structure
chans = [c for c in ALL_CHANS if fits[c]]
fig, axes = plt.subplots(len(chans), 1, figsize=(9, 1.8*len(chans)), squeeze=False)
for row, c in enumerate(chans):
    vals = np.array([fp['nrmse'] for fp in fits[c]])
    vals = vals[np.isfinite(vals) & (vals > 0)]
    ax = axes[row, 0]
    bins = np.logspace(np.log10(max(vals.min(), 1e-3)), np.log10(vals.max()), 60)
    ax.hist(vals, bins=bins, color='steelblue', edgecolor='white', lw=0.3)
    ax.axvline(NRMSE_MAX, color='crimson', lw=1.5, ls='--',
               label=f'cut={NRMSE_MAX}  pass={100*(vals<=NRMSE_MAX).mean():.0f}%')
    ax.set_xscale('log'); ax.legend(fontsize=8, loc='upper right')
    ax.set_title(f'{c}  n={len(vals)}', fontsize=9)
    ax.grid(alpha=0.2); ax.tick_params(labelsize=7)
axes[-1, 0].set_xlabel('NRMSE')
plt.tight_layout(); plt.show()

In [5]:
# fitted-curve fan comparison: before (left) vs after (right) the cut; common pretrigger 16050, peak-normalized
lo, hi = RISE_REF_IDX - 500, RISE_REF_IDX + 5000
x = X_FULL[lo:hi]; t_ms = x / SAMPLERATE * 1e3
rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(chans), 2, figsize=(13, 2.4*len(chans)), squeeze=False)
for row, c in enumerate(chans):
    groups = [(axes[row,0], fits[c], 'all fit_ok'),
              (axes[row,1], [fp for fp in fits[c] if fp['nrmse'] <= NRMSE_MAX],
               f'NRMSE<={NRMSE_MAX}')]
    for ax, fps, label in groups:
        for i in rng.choice(len(fps), min(150, len(fps)), replace=False) if fps else []:
            fp = fps[i]
            y = two_exp_free_pt(x, fp['amp'], fp['t_rise'], fp['t_fall'], 0.0, float(RISE_REF_IDX))
            pk = y.max()
            if pk > 0:
                ax.plot(t_ms, y/pk, lw=0.4, alpha=0.25, color='steelblue')
        ax.set_title(f'{c}  {label}  (n={len(fps)})', fontsize=8)
        ax.grid(alpha=0.2); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

In [6]:
# t_rise / t_fall before/after the cut (does it remove the slow noise population?)
fig, axes = plt.subplots(len(chans), 2, figsize=(12, 2.0*len(chans)), squeeze=False)
for row, c in enumerate(chans):
    for col, key in enumerate(['t_rise', 't_fall']):
        allv  = np.array([fp[key] for fp in fits[c]]) * 1e3
        cutv  = np.array([fp[key] for fp in fits[c] if fp['nrmse'] <= NRMSE_MAX]) * 1e3
        ax = axes[row, col]
        bins = np.logspace(np.log10(max(allv.min(),1e-4)), np.log10(allv.max()), 50)
        ax.hist(allv, bins=bins, color='lightgray', label='all fit_ok')
        ax.hist(cutv, bins=bins, color='steelblue', alpha=0.8, label=f'NRMSE<={NRMSE_MAX}')
        ax.set_xscale('log'); ax.set_title(f'{c} {key} (ms)', fontsize=8)
        ax.legend(fontsize=7); ax.grid(alpha=0.2); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()